# Notebook 03: NLP Skill Extraction Engine
## SkillGraph AI - Canonical Skill Normalization & Fast Extraction

This notebook evaluates the two-tier inverted-index skill extraction pipeline against occupational taxonomies (O*NET & ESCO), including precision, recall, and false-positive prevention.

In [ ]:
import sys
sys.path.append("..")
from backend.app.nlp.skill_extractor import SkillExtractor
import time

extractor = SkillExtractor()
print(f"Skill taxonomy loaded: {len(extractor.taxonomy)} canonical skills registered.")

### 1. Test Single-Document Extraction & Alias Resolution

In [ ]:
sample_text = """
We are seeking a Lead Machine Learning Engineer proficient in Python, PyTorch, and SQL.
The ideal candidate has hands-on experience building neural networks, designing ETL data pipelines in AWS,
deploying containerized models via Docker, and applying natural language processing (NLP).
Excellent communication and teamwork abilities are mandatory.
"""

extracted = extractor.extract(sample_text)
print(f"Extracted {len(extracted)} canonical skills:")
for s in extracted:
    print(f" - {s['canonical_name']} (matched: '{s['matched_phrase']}', category: {s['category']}, confidence: {s['confidence']})")

### 2. High-Throughput Inverted Index Benchmark

In [ ]:
start = time.time()
corpus = [sample_text] * 1000
batch_results = [extractor.extract(text) for text in corpus]
elapsed = time.time() - start
print(f"Extracted skills across 1,000 job descriptions in {elapsed:.2f} seconds ({len(corpus)/elapsed:.1f} docs/sec).")

### 3. Extraction Precision & False-Positive Prevention

In [ ]:
# Test boundary conditions and false-positive prevention
tricky_texts = [
    ("We operate in the US and use C language.", ["C"]),
    ("Can you communicate clearly?", ["Communication"]),
    ("Experience with R programming and R&D projects.", ["R"]),
]

for text, expected in tricky_texts:
    found = [s['canonical_name'] for s in extractor.extract(text)]
    print(f"Text: '{text}' -> Found: {found}")